# Contrastive Dataset for Probing and Activation 

This notebook creates the a contrastive dataset later used for probing and activation steering.
It combines the AdvBench dataset for harmful examples and the Alpaca dataset for harmless examples.  

The dataset already includes the output of Qwen3s layers for the last token of each prompt (chat template used with generic system prompt)

In [ ]:
%uv pip install transformers==5.8.1 peft==0.19.1 datasets==4.8.5 s3fs trl==1.4.0

In [ ]:
import os
import pandas as pd
import datasets as ds
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
SAMPLE_SIZE=520
# How many samples should be taken form each of the original datasets 
# 520 is the dataset size of AdvBench so we just use the complete dataset and combine it with matching count of harmless prompts

device ="cuda"

In [4]:
adv = ds.load_dataset("walledai/AdvBench")
alp = ds.load_dataset("tatsu-lab/alpaca")

In [ ]:
# take a sample of the datasets, select & rename relevant columns
adv_sample = adv["train"].shuffle(2).take(SAMPLE_SIZE).select_columns(["prompt"])
alp_sample = alp["train"].shuffle(2).take(SAMPLE_SIZE).select_columns(["instruction"]).rename_columns({"instruction": "prompt"})

In [ ]:
# add labels to the samples so that we can combine them into a single dataset
adv_labeled = adv_sample.map(lambda x: {"harmful": True})
alp_labeled = alp_sample.map(lambda x: {"harmful": False})

In [7]:
contrastive = ds.concatenate_datasets([adv_labeled, alp_labeled])

In [ ]:
contrastive.push_to_hub("traberph/advbench-alpaca-contrastive")
contrastive

In [ ]:
contrastive = ds.load_dataset("traberph/advbench-alpaca-contrastive")
contrastive

In [ ]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Relevant Docs 

https://huggingface.co/docs/transformers/main_classes/output#model-outputs  
https://huggingface.co/docs/transformers/main_classes/output#transformers.modeling_outputs.BaseModelOutputWithPast

In [ ]:
# helper to extract the hidden states for the last token of a prompt

def extract(prompt):

    # apply chat template for consistency
    input = tokenizer.apply_chat_template([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}],
      add_generation_prompt=True,
      return_tensors="pt"
    ).to(device)

    # model pass
    with torch.no_grad():
        out = model(**input, output_hidden_states=True)
    
    # return all layers for last token
    return torch.stack(
        # [batch 0, last token, every dim]
        [h[0, -1, :] for h in out.hidden_states]
    )

In [ ]:
# apply the helper to the dataset
contrastive_cache = contrastive.map(lambda x: {"layers_last_token": extract(x["prompt"])})

In [ ]:
# push dataset with hidden states to hf hub for later probing and activation steering
contrastive_cache.push_to_hub("traberph/advbench-alpaca-contrastive")